## 🗣️ CTI-Bench MCQ Evaluation — Google Colab (GGUF)

Evaluates `Adxt/Qwen2.5-1.5B-Instruct-CYSECFINETUNED` (GGUF Q8_0) on the CTI-Bench MCQ dataset using **greedy decoding** with GPU acceleration via `llama-cpp-python`.

In [ ]:
# ========================================
# 📦 Install Dependencies
# ========================================
!pip install -q datasets huggingface_hub

# Install llama-cpp-python with CUDA support for GPU inference
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir

In [ ]:
import os
import time
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

### ⚙️ Configuration

In [ ]:
# ========================================
# 🔧 CONFIGURATION
# ========================================

REPO_ID = "Adxt/Qwen2.5-1.5B-Instruct-CYSECFINETUNED"
GGUF_FILENAME = "Qwen2.5-1.5B-q8_0.gguf"
MODEL_NAME = "Qwen2.5-1.5B-CYSECFINETUNED"  # Used for output filenames

# Dataset subset: None = all 2500, or an integer (e.g. 100) for quick test
SUBSET_SIZE = None

# System Prompt
SYSTEM_PROMPT = "You are a cybersecurity expert specializing in cyberthreat intelligence."

# Generation Parameters (greedy decoding)
MAX_TOKENS = 64
SEED = 42

# GGUF settings
GGUF_N_CTX = 2048        # Context window size
GGUF_N_GPU_LAYERS = -1   # -1 = offload ALL layers to GPU

# Save results to Google Drive?
SAVE_TO_DRIVE = True

print(f"✓ Configuration loaded")
print(f"  Repo      : {REPO_ID}")
print(f"  GGUF file : {GGUF_FILENAME}")
print(f"  GPU layers: {GGUF_N_GPU_LAYERS}")
print(f"  Context   : {GGUF_N_CTX}")
print(f"  Subset    : {'ALL (2500)' if SUBSET_SIZE is None else SUBSET_SIZE}")
print(f"  Decoding  : Greedy (temperature=0.0)")

### 📦 Load Dataset

In [ ]:
ds = load_dataset("AI4Sec/cti-bench", "cti-mcq")
data = ds['test'] if 'test' in ds else ds['train']

if SUBSET_SIZE is not None:
    data_subset = data.select(range(min(SUBSET_SIZE, len(data))))
else:
    data_subset = data

print(f"Total examples in dataset : {len(data)}")
print(f"Examples to evaluate      : {len(data_subset)}")
print(f"Column names: {data.column_names}")
print()
print("First example:")
print(data[0])

### 🚀 Download & Load GGUF Model

In [ ]:
# Download GGUF file from HuggingFace
print(f"Downloading {GGUF_FILENAME} from {REPO_ID}...")
gguf_path = hf_hub_download(repo_id=REPO_ID, filename=GGUF_FILENAME)
print(f"✓ Downloaded to: {gguf_path}")
print(f"  File size: {os.path.getsize(gguf_path) / 1e9:.2f} GB")

# Load model with GPU offloading
print(f"\nLoading GGUF model...")
llm = Llama(
    model_path=gguf_path,
    n_ctx=GGUF_N_CTX,
    n_gpu_layers=GGUF_N_GPU_LAYERS,
    seed=SEED,
    verbose=False,
)

print(f"✓ GGUF model loaded: {MODEL_NAME}")
print(f"  GPU layers: {GGUF_N_GPU_LAYERS} (-1 = all)")

### 🧠 Prediction & Evaluation

In [ ]:
def get_single_prediction(question):
    """Generate a prediction using greedy decoding via llama-cpp."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=0.0,  # Greedy decoding
        seed=SEED,
    )
    return response["choices"][0]["message"]["content"].strip()


def format_mcq(text):
    """Extract the A/B/C/D answer from model output."""
    last_line = text.split('\n')[-1].rstrip()

    if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
        return last_line[0]
    if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
        return last_line[-1]
    if last_line.endswith('**'):
        return last_line[-3]
    if len(last_line) == 0:
        last_line = text.split('\n')[-2].rstrip()
        if last_line.startswith('A)') or last_line.startswith('B)') or last_line.startswith('C)') or last_line.startswith('D)'):
            return last_line[0]
        if last_line.endswith('A') or last_line.endswith('B') or last_line.endswith('C') or last_line.endswith('D'):
            return last_line[-1]
        if last_line.endswith('**'):
            return last_line[-3]
    return ' '.join(text.split('\n'))


print("✓ Prediction functions loaded")

In [ ]:
def run_evaluation(task):
    """Run evaluation on CTI-Bench task."""
    print(f"\n{'='*60}")
    print(f"Starting evaluation: {task.upper()}")
    print(f"Model: {MODEL_NAME} (GGUF)")
    print(f"Dataset: AI4Sec/cti-bench")
    print(f"Subset: {len(data_subset)} examples")
    print(f"Decoding: Greedy (temperature=0.0)")
    print(f"{'='*60}\n")

    # Find prompt column
    prompt_column = None
    for col in ['Prompt', 'prompt', 'question', 'input', 'text']:
        if col in data_subset.column_names:
            prompt_column = col
            break

    if prompt_column is None:
        print(f"Available columns: {data_subset.column_names}")
        raise ValueError("Could not find prompt column.")

    print(f"Using column '{prompt_column}' for prompts\n")

    start_time = time.time()
    count_chars = 0
    all_responses = []
    all_results = []
    task_type = task.split('-')[-1]

    for index, example in enumerate(data_subset):
        prompt = example[prompt_column]
        try:
            output = get_single_prediction(prompt)
            count_chars += len(output)
            all_responses.append(output)

            if task_type == 'mcq':
                answer = format_mcq(output)
            else:
                raise ValueError(f'Unknown task type: {task_type}')

        except Exception as e:
            answer = 'Error'
            all_responses.append(answer)
            print(f'❌ Exception at example {index+1}: {e}')

        all_results.append(answer)

        if (index + 1) % 100 == 0:
            elapsed = time.time() - start_time
            print(f"  Progress: {index+1}/{len(data_subset)} ({elapsed:.0f}s)")

    time_taken = time.time() - start_time

    print(f"\n{'='*60}")
    print("EVALUATION COMPLETE")
    print(f"{'='*60}")
    print(f"Time taken: {time_taken:.2f} seconds ({time_taken/60:.2f} minutes)")
    print(f"Characters generated: {count_chars:,}")

    # Save results
    if SAVE_TO_DRIVE:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        output_dir = '/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/results'
    else:
        output_dir = './results'

    os.makedirs(output_dir, exist_ok=True)

    suffix = f"_first{len(data_subset)}" if SUBSET_SIZE is not None else ""
    out_result = f"{output_dir}/{task}_{MODEL_NAME}{suffix}_result.txt"

    with open(out_result, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_results))

    print(f"\n💾 Results saved: {out_result}")
    print(f"{'='*60}\n")

    return all_results


print("✓ Evaluation function ready")

### 🚀 Run Evaluation

In [ ]:
results = run_evaluation('cti-mcq')